# BoshNet Training on RealPC Dataset

This notebook trains **BoshNet** — a Homology Sampler guided neural network for point cloud completion — on the **RealPC** dataset (real-world industrial point cloud pairs).

**Reference:** [Point-Cloud-Completion](https://github.com/stutipathak5/Point-Cloud-Completion/tree/main)

## Architecture
BoshNet uses a PointNet AutoEncoder:
- **Encoder:** 5 Conv1D layers (3→64→128→256→512→128) + global max pooling
- **Latent:** 128-dimensional bottleneck
- **Decoder:** 4 FC layers (128→512→512→1024→output\_size×3)

## RealPC Dataset
- ~40,000 partial/complete point cloud pairs
- 21 industrial object categories from railway environments
- Files: `part_tr.npy`, `comp_tr.npy` (train) and `part_te.npy`, `comp_te.npy` (test)

## 1. Install Dependencies

Install the required packages. **pytorch3d** requires a CUDA-aware build; we install the prebuilt wheel that matches the Colab CUDA / PyTorch version.

In [ ]:
import sys
import torch

# Verify GPU is available
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Install open3d, torchsummary, tqdm, h5py
!pip install -q open3d torchsummary tqdm h5py

# Install pytorch3d — use the prebuilt wheel for the current PyTorch + CUDA version
import torch
pyt_version_str = torch.__version__.split("+")[0].replace(".", "")
version_str = "".join([
    f"py3{sys.version_info.minor}_cu",
    torch.version.cuda.replace(".", ""),
    f"_pyt{pyt_version_str}"
])
print(f"Attempting to install pytorch3d wheel for: {version_str}")

!pip install -q fvcore iopath
try:
    !pip install -q --no-index --no-cache-dir pytorch3d \
        -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{version_str}/download.html
    import pytorch3d
    print("pytorch3d installed successfully:", pytorch3d.__version__)
except Exception as e:
    print(f"Prebuilt wheel not available ({e}). Falling back to source build (slower)...")
    !pip install -q 'git+https://github.com/facebookresearch/pytorch3d.git@stable'

## 2. Mount Google Drive and Download the RealPC Dataset

The RealPC dataset is hosted on Google Drive. Mount your Drive and either:
- **Option A (recommended):** Place the dataset files directly in your Drive under `MyDrive/realpc/`
- **Option B:** Use `gdown` to download from the shared link

> **Dataset link:** https://drive.google.com/file/d/1decE9LodqeZP1C4zIxgqXBgvDZWD87j2/view?usp=sharing

Expected files after extraction:
- `part_tr.npy` — partial point clouds (training)
- `comp_tr.npy` — complete point clouds (training)
- `part_te.npy` — partial point clouds (test)
- `comp_te.npy` — complete point clouds (test)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Configure paths ──────────────────────────────────────────────────────────
# Change DATA_DIR to the folder that contains part_tr.npy / comp_tr.npy
DATA_DIR = '/content/drive/MyDrive/realpc'
LOG_DIR  = '/content/drive/MyDrive/realpc_logs'
os.makedirs(LOG_DIR, exist_ok=True)

# If the dataset is not yet in your Drive, download it here:
# (uncomment and run the lines below)
# ─────────────────────────────────────────────────────────────────────────────
# !pip install -q gdown
# import gdown
# os.makedirs(DATA_DIR, exist_ok=True)
# GDRIVE_FILE_ID = '1decE9LodqeZP1C4zIxgqXBgvDZWD87j2'
# gdown.download(f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}',
#                '/content/realpc.zip', quiet=False)
# !unzip -q /content/realpc.zip -d {DATA_DIR}

# Verify files exist
for fname in ['part_tr.npy', 'comp_tr.npy']:
    path = os.path.join(DATA_DIR, fname)
    assert os.path.exists(path), f"Missing dataset file: {path}"
    print(f"Found: {path}")

## 3. Model Definition (BoshNet / PointCloudAE)

Exact reproduction of `model.py` from the [original repository](https://github.com/stutipathak5/Point-Cloud-Completion/blob/main/BOSHNet/model.py).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PointCloudAE(nn.Module):
    """PointNet AutoEncoder used by BoshNet.

    Input : (B, 3, N)  — batch of partial point clouds
    Output: (B, 3, M)  — batch of completed point clouds
    """

    def __init__(self, input_size: int, output_size: int, latent_size: int = 128):
        super().__init__()
        self.latent_size = latent_size
        self.input_size  = input_size
        self.output_size = output_size

        # Encoder
        self.conv1 = nn.Conv1d(3,   64,  1)
        self.conv2 = nn.Conv1d(64,  128, 1)
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.conv4 = nn.Conv1d(256, 512, 1)
        self.conv5 = nn.Conv1d(512, self.latent_size, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(self.latent_size)

        # Decoder
        self.dec1 = nn.Linear(self.latent_size, 512)
        self.dec2 = nn.Linear(512, 512)
        self.dec3 = nn.Linear(512, 1024)
        self.dec4 = nn.Linear(1024, self.output_size * 3)

    def encoder(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.bn5(self.conv5(x))
        x = torch.max(x, 2, keepdim=True)[0]
        return x.view(-1, self.latent_size)

    def decoder(self, x):
        x = F.relu(self.dec1(x))
        x = F.relu(self.dec2(x))
        x = F.relu(self.dec3(x))
        x = self.dec4(x)
        return x.view(-1, 3, self.output_size)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


print("PointCloudAE (BoshNet) defined.")

## 4. Dataset Utilities

Exact reproduction of the relevant parts of `utils.py` from the [original repository](https://github.com/stutipathak5/Point-Cloud-Completion/blob/main/BOSHNet/utils.py).

In [ ]:
import numpy as np
import open3d as o3d
import shutil
from torch.utils.data import Dataset


def normalize(pc_array: np.ndarray) -> np.ndarray:
    """Normalize each point cloud in the array to a unit bounding box.

    Steps:
      1. Compute axis-aligned bounding box and centre it.
      2. Estimate yaw angle and apply rotation to align with axes.
      3. Scale to unit length along the principal axis.
      4. Swap y/z axes so that z points up.
    """
    for i in range(pc_array.shape[0]):
        ptcloud = pc_array[i]
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(ptcloud)
        bbox = pcd.get_axis_aligned_bounding_box()
        corners = np.asarray(bbox.get_box_points())
        corners[[3, 7]] = corners[[7, 3]]
        center = (corners.min(0) + corners.max(0)) / 2
        corners -= center
        yaw = np.arctan2(corners[3, 1] - corners[0, 1],
                         corners[3, 0] - corners[0, 0])
        rotation = np.array([
            [np.cos(yaw), -np.sin(yaw), 0],
            [np.sin(yaw),  np.cos(yaw), 0],
            [0,            0,           1]
        ])
        corners = np.dot(corners, rotation)
        scale = corners[3, 0] - corners[0, 0]
        if scale == 0:
            scale = 1e-4
        ptcloud = np.dot(ptcloud - center, rotation) / scale
        # Swap y and z so z points up
        ptcloud = np.dot(ptcloud, [[1, 0, 0], [0, 0, 1], [0, 1, 0]])
        pc_array[i] = ptcloud
    return pc_array


class CatenaryLoader(Dataset):
    """PyTorch Dataset for paired partial / complete point clouds.

    __len__ returns (N - 1) so that __getitem__ can safely return
    ``self.partial[index + 1]`` as the fourth element.  That extra value
    was designed for triplet / contrastive losses (commented out in the
    original training script) and is discarded with ``_`` in the training
    functions below.
    """

    def __init__(self, partial: np.ndarray, complete: np.ndarray):
        super().__init__()
        self.partial  = partial
        self.complete = complete

    def __len__(self):
        # N-1: keeps index+1 access in __getitem__ within bounds
        return self.complete.shape[0] - 1

    def __getitem__(self, index):
        # Fourth value (next partial) retained for API compatibility;
        # it is unused by the current training loop.
        return index, self.partial[index], self.complete[index], self.partial[index + 1]


def clear_folder(path: str):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.mkdir(path)


print("Dataset utilities defined.")


## 5. Load and Preprocess the RealPC Dataset

In [ ]:
import numpy as np
from tqdm import trange

print("Loading dataset...")
partial_data  = np.load(os.path.join(DATA_DIR, 'part_tr.npy'))  # (N, 4000, 3)
complete_data = np.load(os.path.join(DATA_DIR, 'comp_tr.npy'))  # (N, 5000, 3)

print(f"Partial  shape : {partial_data.shape}")
print(f"Complete shape : {complete_data.shape}")

print("Normalizing point clouds (this may take a few minutes)...")
partial_data  = normalize(partial_data).astype(np.float32)
complete_data = normalize(complete_data).astype(np.float32)

print("Normalisation complete.")

## 6. Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader

# ── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE  = 32    # reduce if you run out of GPU memory
NUM_EPOCHS  = 1500
LR          = 1e-3
LATENT_SIZE = 128
NUM_WORKERS = 2
# ─────────────────────────────────────────────────────────────────────────────

n_samples  = partial_data.shape[0]
input_size  = partial_data.shape[1]   # number of points in partial cloud
output_size = complete_data.shape[1]  # number of points in complete cloud

split = int(n_samples * 0.8)

train_dataset = CatenaryLoader(partial_data[:split],  complete_data[:split])
test_dataset  = CatenaryLoader(partial_data[split:],  complete_data[split:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, drop_last=False)

print(f"Train samples : {len(train_dataset)}  |  batches : {len(train_loader)}")
print(f"Test  samples : {len(test_dataset)}   |  batches : {len(test_loader)}")
print(f"Input  size   : {input_size}")
print(f"Output size   : {output_size}")

## 7. Initialise Model, Optimizer, and Loss

In [ ]:
import torch.optim as optim
from torchsummary import summary
from pytorch3d.loss import chamfer_distance

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

net = PointCloudAE(input_size, output_size, LATENT_SIZE).to(device)
optimizer = optim.Adam(net.parameters(), lr=LR)

# Print model summary (use the partial-cloud size as input)
summary(net, (3, input_size), device=str(device))

## 8. (Optional) Resume from Checkpoint

If you have a previously saved checkpoint, set `CHECKPOINT_PATH` to its path and run this cell before training.

In [ ]:
CHECKPOINT_PATH = ''  # e.g. '/content/drive/MyDrive/realpc_logs/ckpt_best_300.t7'
start_epoch = 0

if CHECKPOINT_PATH:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    net.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch = checkpoint['epoch']
    print(f"Resumed from checkpoint: {CHECKPOINT_PATH}")
    print(f"Starting from epoch: {start_epoch}")
else:
    print("No checkpoint provided — training from scratch.")

## 9. Training and Evaluation Functions

BoshNet's key insight is to pass **multiple sparse sub-samplings** of both the partial *and* complete inputs through the network during training. This guides the encoder to learn topologically consistent latent representations from the very first epoch.

In [ ]:
import time


def train_epoch(net, optimizer, loader, device):
    """Run one training epoch and return the average Chamfer loss."""
    net.train()
    epoch_loss = 0.0
    num_batches = 0

    for _, incomplete, complete, _ in loader:
        incomplete = incomplete.to(device)  # (B, N_partial,  3)
        complete   = complete.to(device)    # (B, N_complete, 3)

        # ── Sparse sub-samples from the partial cloud ──────────────────────
        sparse_seeds_partial = [
            incomplete[:, ::4],
            incomplete[:, ::8],
            incomplete[:, ::16],
            incomplete[:, ::32],
            incomplete[:, ::64],
        ]

        # ── Sparse sub-samples from the complete cloud ──────────────────────
        sparse_seeds_complete = [
            complete[:, ::4],
            complete[:, ::8],
            complete[:, ::16],
            complete[:, ::32],
            complete[:, ::64],
        ]

        optimizer.zero_grad()

        # ── Forward pass: partial and complete inputs ────────────────────────
        out_incomplete, _ = net(incomplete.permute(0, 2, 1))
        out_complete,   _ = net(complete.permute(0, 2, 1))

        cham_incomplete, _ = chamfer_distance(complete, out_incomplete.permute(0, 2, 1))
        cham_complete,   _ = chamfer_distance(complete, out_complete.permute(0, 2, 1))

        # ── Forward pass: sparse sub-samples of the partial cloud ────────────
        cham_partial_sparse = []
        for seed in sparse_seeds_partial:
            out, _ = net(seed.permute(0, 2, 1))
            loss, _ = chamfer_distance(complete, out.permute(0, 2, 1))
            cham_partial_sparse.append(loss)

        # ── Forward pass: sparse sub-samples of the complete cloud ───────────
        cham_complete_sparse = []
        for seed in sparse_seeds_complete:
            out, _ = net(seed.permute(0, 2, 1))
            loss, _ = chamfer_distance(complete, out.permute(0, 2, 1))
            cham_complete_sparse.append(loss)

        # ── Combined loss ────────────────────────────────────────────────────
        num_partial_terms  = 2 + len(cham_partial_sparse)   # incomplete + complete + sparse partial
        num_complete_terms = len(cham_complete_sparse)

        loss_partial  = (cham_incomplete + cham_complete + sum(cham_partial_sparse)) / num_partial_terms
        loss_complete = sum(cham_complete_sparse) / num_complete_terms
        loss = loss_partial + loss_complete

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / max(num_batches, 1)


@torch.no_grad()
def test_epoch(net, loader, device):
    """Evaluate on the test set and return the average Chamfer loss."""
    net.eval()
    epoch_loss = 0.0
    num_batches = 0

    for _, incomplete, complete, _ in loader:
        incomplete = incomplete.to(device)
        complete   = complete.to(device)
        output, _  = net(incomplete.permute(0, 2, 1))
        loss, _    = chamfer_distance(complete, output.permute(0, 2, 1))
        epoch_loss  += loss.item()
        num_batches += 1

    return epoch_loss / max(num_batches, 1)


print("Training and evaluation functions defined.")

## 10. Training Loop

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Colab
import matplotlib.pyplot as plt

train_loss_list = []
test_loss_list  = []
best_test_loss  = float('inf')

os.makedirs(LOG_DIR, exist_ok=True)
log_file = os.path.join(LOG_DIR, 'training_log.txt')

print(f"Training BoshNet for {NUM_EPOCHS} epochs ...")
print(f"Checkpoints and logs will be saved to: {LOG_DIR}")

for epoch in trange(start_epoch, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss = train_epoch(net, optimizer, train_loader, device)
    test_loss  = test_epoch(net,  test_loader,  device)

    train_loss_list.append(train_loss)
    test_loss_list.append(test_loss)

    epoch_time = time.time() - t0

    # ── Save best checkpoint ─────────────────────────────────────────────────
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        state = {'epoch': epoch + 1,
                 'state_dict': net.state_dict(),
                 'optimizer':  optimizer.state_dict()}
        torch.save(state, os.path.join(LOG_DIR, f'ckpt_best_{epoch}.t7'))
        print(f"  ↳ New best at epoch {epoch}: test_loss={test_loss:.6f}")

    # ── Save periodic checkpoint every 15 epochs ─────────────────────────────
    if epoch % 15 == 0:
        state = {'epoch': epoch + 1,
                 'state_dict': net.state_dict(),
                 'optimizer':  optimizer.state_dict()}
        torch.save(state, os.path.join(LOG_DIR, f'ckpt_{epoch}.t7'))

    # ── Log to file ──────────────────────────────────────────────────────────
    log_line = (f"epoch {epoch}  train_loss: {train_loss:.6f}  "
                f"test_loss: {test_loss:.6f}  time: {epoch_time:.1f}s")
    with open(log_file, 'a') as f:
        f.write(log_line + '\n')

    # ── Update loss plot every 10 epochs ─────────────────────────────────────
    if epoch % 10 == 0:
        plt.figure(figsize=(8, 4))
        plt.plot(train_loss_list, label='Train')
        plt.plot(test_loss_list,  label='Test')
        plt.xlabel('Epoch')
        plt.ylabel('Chamfer Distance Loss')
        plt.title('BoshNet Training on RealPC')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(LOG_DIR, 'loss.png'))
        plt.close()

print("\nTraining complete!")
print(f"Best test loss : {best_test_loss:.6f}")

## 11. Plot Final Training Curves

In [ ]:
import matplotlib
matplotlib.use('inline')  # switch to interactive backend for display
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_loss_list, label='Train Chamfer Loss')
plt.plot(test_loss_list,  label='Test  Chamfer Loss')
plt.xlabel('Epoch')
plt.ylabel('Chamfer Distance Loss')
plt.title('BoshNet — Training on RealPC Dataset')
plt.legend()
plt.tight_layout()
plt.show()

## 12. Visualise Point Cloud Completions

Visualise a few examples from the test set: **partial input** (left), **ground-truth complete** (centre), and **BoshNet prediction** (right).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # required for 3D projection

net.eval()

# Grab one batch from the test set
sample_iter = iter(test_loader)
_, partial_batch, complete_batch, _ = next(sample_iter)

with torch.no_grad():
    pred_batch, _ = net(partial_batch.to(device).permute(0, 2, 1))
    pred_batch = pred_batch.permute(0, 2, 1).cpu().numpy()

partial_np  = partial_batch.numpy()
complete_np = complete_batch.numpy()

NUM_VIS = min(4, partial_np.shape[0])

fig = plt.figure(figsize=(15, NUM_VIS * 3))
for idx in range(NUM_VIS):
    for col, (pc, title) in enumerate([
        (partial_np[idx],  'Partial Input'),
        (complete_np[idx], 'Ground Truth'),
        (pred_batch[idx],  'BoshNet Prediction'),
    ]):
        ax = fig.add_subplot(NUM_VIS, 3, idx * 3 + col + 1, projection='3d')
        ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2],
                   c='steelblue', marker='.', alpha=0.6, s=4)
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_zlim(-1, 1)
        ax.set_title(f"{title} (sample {idx})", fontsize=9)
        ax.axis('off')

plt.suptitle('BoshNet — Point Cloud Completion on RealPC', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'completions_preview.png'), dpi=150)
plt.show()
